# Live Change — watch mirroring replicate in real time

The proof moment: change a row **in the source Azure SQL database**, then watch
the change arrive in Fabric's replicated copy — with no pipeline, no refresh,
and no code in between.

1. Read a product's price from the **mirrored** table (Fabric copy)
2. `UPDATE` that price **in Azure SQL** (the source)
3. Poll the mirrored table until the new price appears, timing the replication

> Authentication is **Microsoft Entra-only** — this notebook runs as you (the
> server's Entra admin) and acquires a SQL access token; there is no password.

In [ ]:
SQL_SERVER = "{{SQL_SERVER}}"
SQL_DATABASE = "{{SQL_DATABASE}}"
WORKSPACE_ID = "{{WORKSPACE_ID}}"
MIRRORED_DB_ID = "{{MIRRORED_DB_ID}}"

# Microsoft Entra-only authentication — acquire a SQL access token for you.
import notebookutils

def _sql_access_token():
    errors = []
    for audience in ("https://database.windows.net/", "https://database.windows.net"):
        try:
            tok = notebookutils.credentials.getToken(audience)
            if tok:
                return tok
        except Exception as e:  # noqa: BLE001
            errors.append(f"{audience}: {e}")
    raise RuntimeError("Could not acquire an Azure SQL access token via "
                       "notebookutils.credentials.getToken — " + "; ".join(errors))

ACCESS_TOKEN = _sql_access_token()

JDBC_URL = (
    f"jdbc:sqlserver://{SQL_SERVER}:1433;database={SQL_DATABASE};"
    "encrypt=true;trustServerCertificate=false;loginTimeout=60;"
)

def mirrored(table: str):
    path = (
        f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/"
        f"{MIRRORED_DB_ID}/Tables/dbo/{table}"
    )
    return spark.read.format("delta").load(path)

def run_sql(statement: str):
    """Execute a statement against the SOURCE Azure SQL database (Entra auth)."""
    jvm = spark._sc._jvm
    props = jvm.java.util.Properties()
    props.setProperty("accessToken", ACCESS_TOKEN)
    jvm.java.lang.Class.forName("com.microsoft.sqlserver.jdbc.SQLServerDriver")
    conn = jvm.java.sql.DriverManager.getConnection(JDBC_URL, props)
    stmt = conn.createStatement()
    n = stmt.executeUpdate(statement)
    stmt.close()
    conn.close()
    return n

print("Ready")

In [ ]:
# Step 1 — pick a product and read its CURRENT price from the Fabric copy
TARGET_SKU = "SKU-0001"

row = mirrored("products").filter(f"sku = '{TARGET_SKU}'").collect()[0]
old_cost = float(row["unit_cost"])
print(f"{row['product_name']} ({TARGET_SKU})")
print(f"Price in the MIRRORED copy right now: {old_cost:.2f}")

In [ ]:
# Step 2 — change the price IN THE SOURCE database (Azure SQL), not in Fabric
new_cost = round(old_cost + 25.00, 2)
n = run_sql(f"UPDATE dbo.products SET unit_cost = {new_cost} WHERE sku = '{TARGET_SKU}'")
print(f"UPDATE executed against Azure SQL ({n} row affected)")
print(f"  source price is now : {new_cost:.2f}")

still = float(mirrored("products").filter(f"sku = '{TARGET_SKU}'").collect()[0]["unit_cost"])
print(f"  mirrored copy shows : {still:.2f}   <- not replicated yet")

In [ ]:
# Step 3 — watch Fabric catch up BY ITSELF (no pipeline, no refresh)
import time

POLL_EVERY = 10      # seconds
MAX_WAIT = 600       # mirroring usually lands in well under a minute

start = time.time()
replicated = None
while time.time() - start < MAX_WAIT:
    current = float(mirrored("products").filter(f"sku = '{TARGET_SKU}'").collect()[0]["unit_cost"])
    elapsed = int(time.time() - start)
    if abs(current - new_cost) < 0.005:
        replicated = elapsed
        print(f"[{elapsed:>3}s] mirrored copy: {current:.2f}   REPLICATED in ~{elapsed}s")
        break
    print(f"[{elapsed:>3}s] mirrored copy: {current:.2f}   (waiting for replication...)")
    time.sleep(POLL_EVERY)

if replicated is None:
    print(f"Change not visible after {MAX_WAIT}s. First sync after seeding can lag a few")
    print("minutes - check the Mirrored Database's replication status page, then re-run.")

In [ ]:
# Punchline — and it works for new rows too
import uuid

txn_id = f"TXN-LIVE-{uuid.uuid4().hex[:8].upper()}"
run_sql(
    "INSERT INTO dbo.pos_transactions "
    "(transaction_id, store_id, product_id, transaction_timestamp, quantity, unit_price, discount_pct) "
    f"VALUES ('{txn_id}', 'STR-001', '{TARGET_SKU}', SYSUTCDATETIME(), 3, {new_cost}, 0)"
)
print(f"Inserted a brand-new sale in Azure SQL: {txn_id}")

start = time.time()
while time.time() - start < MAX_WAIT:
    hit = mirrored("pos_transactions").filter(f"transaction_id = '{txn_id}'").count()
    elapsed = int(time.time() - start)
    if hit:
        print(f"[{elapsed:>3}s] new sale visible in Fabric - replicated in ~{elapsed}s")
        break
    print(f"[{elapsed:>3}s] not in Fabric yet...")
    time.sleep(POLL_EVERY)

print("\nNothing moved this data except Fabric Mirroring itself:")
print("  no pipeline - no schedule - no refresh - no code")